# STA272 Lab #4

## Purpose of Lab

Welcome to your fourth weekly tutorial lab!

You are encouraged to refer to lecture content and liberally use course resources.

## Logistics

Due date: The homework is due **11:59pm on Thursday, January 29, 2026.**

You will submit your homework on [MarkUs](https://markus.teach.cs.toronto.edu/markus/).

1. Download this file (`STA272_lab4_student.ipynb`) from JupyterHub. (See [our JupyterHub Guide](../guides/jupyterhub_guide.ipynb) for detailed instructions.)
2. Submit this file to MarkUs under the hw4 assignment. (See [our MarkUs Guide](../guides/markus_guide.ipynb) for detailed instructions.)

## Logistic Regression for Classification

In this lab, we will practice building and evaluating a logistic regression model using the movie dataset.

**Research Question:** Can we predict whether a movie will be highly rated based on its characteristics?

We will define a movie as "highly rated" if its `vote_average` is 7.0 or higher on TMDB's 10-point scale.

### Task #1

Read `../Lectures/movies.csv` into a `pandas` dataframe called `movies`.

The code cell below contains starter code. Fill in the ellipsis `(...)` with the required code.

In [ ]:
# fill in your answer for Task #1 in this cell

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

pd.set_option("display.float_format", "{:.2f}".format)

# Load the data
movies = pd.read_csv(...)

# check your work
print(f"Dataset shape: {movies.shape}")
movies.head()

### Task #2

Create a binary outcome variable called `is_highly_rated` that equals 1 if `vote_average >= 7.0` and 0 otherwise.

Hint: Use a comparison operator and `.astype(int)` to convert boolean to integer.

In [ ]:
# fill in your answer for Task #2 in this cell

movies['is_highly_rated'] = (movies['vote_average'] >= ...).astype(...)

# check your work
print("Distribution of outcome:")
print(movies['is_highly_rated'].value_counts())
print(f"\nPercentage of highly rated movies: {movies['is_highly_rated'].mean()*100:.1f}%")

### Task #3

Create a bar plot showing the count of highly rated (1) vs. not highly rated (0) movies.

Hint: Use `.value_counts()` and `.plot(kind='bar')`.

In [ ]:
# fill in your answer for Task #3 in this cell

counts = movies['is_highly_rated'].value_counts().reindex([1, 0])

ax = counts.plot(kind=..., color=["#228ab7", "#e74c3c"])
ax.set_xticklabels(['Highly Rated (1)', 'Not Highly Rated (0)'], rotation=0)
plt.xlabel('Outcome')
plt.ylabel('Count')
plt.title('Distribution of Movie Ratings')
plt.show()

### Task #4

Create a histogram of the `popularity` column to examine its distribution.

Hint: Use `plt.hist()` with `bins=50`.

In [ ]:
# fill in your answer for Task #4 in this cell

plt.figure(figsize=(10, 5))
plt.hist(movies[...], bins=..., edgecolor='black', alpha=0.7)
plt.xlabel('Popularity Score')
plt.ylabel('Frequency')
plt.title('Distribution of Popularity Scores')
plt.show()

# Summary statistics
print("Summary statistics for popularity:")
print(movies['popularity'].describe())

### Task #5

The popularity distribution is heavily **right-skewed**. Create a log-transformed version of popularity called `log_popularity` using `np.log(popularity + 1)`.

Then create side-by-side histograms comparing the original and log-transformed distributions.

In [ ]:
# fill in your answer for Task #5 in this cell

# Create log-transformed popularity
movies['log_popularity'] = np.log(movies['popularity'] + ...)

# Compare distributions side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original popularity
axes[0].hist(movies['popularity'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Popularity Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Original Popularity (Right-Skewed)')

# Log-transformed popularity
axes[1].hist(movies[...], bins=50, edgecolor='black', alpha=0.7, color='darkorange')
axes[1].set_xlabel('Log(Popularity + 1)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Log-Transformed Popularity (More Symmetric)')

plt.tight_layout()
plt.show()

### Task #6

Create a boxplot comparing `log_popularity` between highly rated and not highly rated movies.

Hint: Use `movies.boxplot(column=..., by=...)`.

In [ ]:
# fill in your answer for Task #6 in this cell

movies.boxplot(column=..., by=...)
plt.xlabel('Is Highly Rated')
plt.ylabel('Log(Popularity + 1)')
plt.title('Log Popularity Distribution by Rating Category')
plt.suptitle('')
plt.show()

**Question:** Do you see evidence of a relationship between log popularity and whether a movie is highly rated?

*Your answer here:*



### Task #7

Prepare the data for logistic regression:

1. Filter to movies with `budget > 0` and create `movies_clean`
2. Create `budget_millions` by dividing budget by 1,000,000
3. Create genre indicator variables for Action, Comedy, Drama, and Horror
4. Split the data into training (80%) and test (20%) sets with `random_state=42`

In [ ]:
# fill in your answer for Task #7 in this cell

# Create budget in millions (handle zeros as missing)
movies_clean = movies[movies['budget'] > 0].copy()
movies_clean['budget_millions'] = movies_clean['budget'] / ...

# Create log-transformed popularity for the cleaned data
movies_clean['log_popularity'] = np.log(movies_clean['popularity'] + 1)

# Create genre indicators
genres = ['Action', 'Comedy', 'Drama', 'Horror']
for genre in genres:
    movies_clean[genre.lower()] = movies_clean['genres'].fillna('').str.contains(...).astype(int)

print(f"Movies with valid budget: {len(movies_clean)}")

In [ ]:
# fill in your answer for Task #7 (continued) in this cell

# Define features
feature_cols = ['budget_millions', 'log_popularity', 'runtime', 'action', 'comedy', 'drama', 'horror']

# Recreate the outcome variable for the cleaned data
movies_clean['is_highly_rated'] = (movies_clean['vote_average'] >= 7.0).astype(int)

X = movies_clean[feature_cols]
y = movies_clean['is_highly_rated']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=..., random_state=..., stratify=y
)

print(f"Training set: {len(X_train)} movies")
print(f"Test set: {len(X_test)} movies")

### Task #8

Fit a logistic regression model using `statsmodels` on the training data.

Display the coefficients, odds ratios (exponentiated coefficients), and p-values.

Hint: Remember to add a constant using `sm.add_constant()` and use `sm.Logit()` instead of `sm.OLS()`.

In [ ]:
# fill in your answer for Task #8 in this cell

# Fit logistic regression
X_train_const = sm.add_constant(...)
model = sm.Logit(..., ...).fit()

# Display coefficients and odds ratios
results = pd.DataFrame({
    'Coefficient': model.params,
    'Odds Ratio': np.exp(model.params),
    'P-value': model.pvalues
})
print(results)

**Question:** Which predictors are statistically significant (p-value < 0.05)? Interpret the odds ratio for `log_popularity`.

*Your answer here:*



### Task #9

Make predictions on the test set and compute the confusion matrix and classification metrics (accuracy, precision, recall, F1).

Use a threshold of 0.5 to convert predicted probabilities to class predictions.

In [ ]:
# fill in your answer for Task #9 in this cell

# Make predictions on test set
X_test_const = sm.add_constant(...)
y_pred_prob = model.predict(...)
y_pred = (y_pred_prob >= ...).astype(int)

# Confusion matrix
cm = confusion_matrix(..., ...)
print("Confusion Matrix:")
print(f"                    Predicted")
print(f"                    Not HR   HR")
print(f"Actual Not HR       {cm[0,0]:5d}  {cm[0,1]:5d}")
print(f"       Highly Rated {cm[1,0]:5d}  {cm[1,1]:5d}")

# Metrics
print(f"\nClassification Metrics:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")

**Question:** What does the recall metric tell us about the model's performance?

*Your answer here:*



### Task #10

Create a ROC curve and calculate the AUC (Area Under the Curve).

Hint: Use `roc_curve()` and `roc_auc_score()` from sklearn.metrics.

In [ ]:
# fill in your answer for Task #10 in this cell

from sklearn.metrics import roc_curve, roc_auc_score

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(..., ...)
auc = roc_auc_score(..., ...)

# Plot
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC Curve (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve for Predicting Highly Rated Movies')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"AUC = {auc:.3f}")

**Question:** What does an AUC of this value tell us about the model's discriminative ability?

*Your answer here:*

